# Repeat-until-success $R_y(\theta)$ over $\{H,\,X,\,Z,\,\mathrm{CCX}\}$

Pick an angle and a precision, get a gate sequence over that set only, on
1 data qubit (`q0`) + 2 ancillas (`q1`,`q2`) that are measured at the end of each round.

Every measurement outcome applies an **exact** rotation by a known angle, so a
non-success outcome is not a failure: the loop simply retargets the remainder.
Costs about $3\times$ fewer gates than the deterministic circuit.

Needs `rysynth.py` and `rysynth_rus.py` next to this notebook.

In [14]:
import math, random, sys, pathlib, time
from fractions import Fraction
import numpy as np

sys.path.insert(0, str(pathlib.Path.cwd()))
import rysynth as R
import rysynth_rus as U

print('loaded', R.__file__)
print('loaded', U.__file__)

loaded /home/dustinseboldt/Desktop/iqpe/h+toff synthesis/rysynth.py
loaded /home/dustinseboldt/Desktop/iqpe/h+toff synthesis/rysynth_rus.py


## Input

* `SEED = None` draws a fresh random angle, an integer reproduces one.
* `THETA = None` uses the random draw; set a number to synthesise that exact angle.
* `EPS` is the angle accuracy of the successful branch.
* `PMIN` is the floor on success probability per round (higher = more gates per round,
  fewer rounds). 0.5 is a good default.

In [15]:
SEED  = None
THETA = None          # e.g. math.pi/7 ; None -> random in [0, 4*pi)
EPS   = 1e-2
PMIN  = 0.5

rng = random.Random(SEED)
if THETA is None:
    THETA = rng.uniform(0.0, 4.0 * math.pi)     # R_y has period 4*pi

print('theta = %.12f rad  (%.6f deg)' % (THETA, math.degrees(THETA)))
print('eps   = %.1e     pmin = %.2f' % (EPS, PMIN))

theta = 3.786589509110 rad  (216.955598 deg)
eps   = 1.0e-02     pmin = 0.50


## Synthesis of one round

In [16]:
t0  = time.time()
res = U.synthesize_rus(THETA, EPS, pmin=PMIN)
print('done in %.2f s' % (time.time() - t0))
print('lde k        : %d   (~log2(1/eps) = %.1f)' % (res['k'], math.log2(1/EPS)))
print('gate count   :', res['counts'])
print('p(success)   : %s = %.6f' % (res['p_success'], float(res['p_success'])))
print('angle error  : %.3e   (budget %.1e)' % (res['angle_err'], EPS))
print('v0           :', res['v0'])
print('v1           :', res['v1'])
a, b = res['v0'][0], res['v0'][1]
print('success Kraus: [[%d,%d],[%d,%d]] / sqrt(2)^%d,  a^2+b^2 = %d'
      % (a, -b, b, a, res['k'], a*a + b*b))

done in 0.01 s
lde k        : 4   (~log2(1/eps) = 6.6)
gate count   : {'X': 18, 'H': 8, 'CCX': 24, 'total': 50}
p(success)   : 5/8 = 0.625000
angle error  : 1.496e-03   (budget 1.0e-02)
v0           : [-1, 3, 2, 1, 1, 0, 0, 0]
v1           : [-3, -1, -1, 2, 0, 1, 0, 0]
success Kraus: [[-1,-3],[3,-1]] / sqrt(2)^4,  a^2+b^2 = 10


## Output: the gate sequence

Time order, left to right. Tuples are `('H',t)`, `('X',t)`, `('Z',t)`, `('CCX',c1,c2,t)`.

In [17]:
def show(gates, head=30):
    for i, g in enumerate(gates[:head]):
        print('%4d  %-4s %s' % (i, g[0], ', '.join('q%d' % q for q in g[1:])))
    if len(gates) > head:
        print('      ...  (%d more)' % (len(gates) - head))

show(res['gates'])

   0  X    q2
   1  X    q1
   2  H    q2
   3  CCX  q0, q1, q2
   4  H    q2
   5  X    q0
   6  H    q2
   7  CCX  q0, q1, q2
   8  H    q2
   9  X    q2
  10  X    q1
  11  CCX  q0, q1, q2
  12  X    q2
  13  CCX  q0, q2, q1
  14  X    q2
  15  CCX  q0, q1, q2
  16  X    q0
  17  H    q1
  18  X    q2
  19  CCX  q1, q2, q0
  20  X    q2
  21  CCX  q0, q2, q1
  22  CCX  q1, q2, q0
  23  CCX  q0, q2, q1
  24  H    q2
  25  X    q2
  26  X    q1
  27  CCX  q1, q2, q0
  28  X    q2
  29  X    q1
      ...  (20 more)


In [18]:
print(R.print_circuit(res['gates'], 25))

X(q2) X(q1) H(q2) CCX(q0,q1,q2) H(q2) X(q0) H(q2) CCX(q0,q1,q2) H(q2) X(q2) X(q1) CCX(q0,q1,q2) X(q2) CCX(q0,q2,q1) X(q2) CCX(q0,q1,q2) X(q0) H(q1) X(q2) CCX(q1,q2,q0) X(q2) CCX(q0,q2,q1) CCX(q1,q2,q0) CCX(q0,q2,q1) H(q2) ... (+25 more)


In [6]:
# OpenQASM 2.0 for ONE round, with the ancilla measurement.
# The repeat/retarget loop itself is host-side classical control.
def to_qasm(gates):
    out = ['OPENQASM 2.0;', 'include "qelib1.inc";', 'qreg q[3];', 'creg c[2];']
    for g in gates:
        out.append('ccx q[%d],q[%d],q[%d];' % g[1:] if g[0] == 'CCX'
                   else '%s q[%d];' % (g[0].lower(), g[1]))
    out += ['measure q[1] -> c[0];', 'measure q[2] -> c[1];']
    return '\n'.join(out)

qasm = to_qasm(res['gates'])
pathlib.Path('ry_rus_round.qasm').write_text(qasm)
print('\n'.join(qasm.splitlines()[:10])); print('...')
print('\n'.join(qasm.splitlines()[-2:]))
print('\nwritten to ry_rus_round.qasm (%d lines)' % len(qasm.splitlines()))

OPENQASM 2.0;
include "qelib1.inc";
qreg q[3];
creg c[2];
x q[2];
x q[1];
h q[2];
ccx q[0],q[1],q[2];
h q[2];
x q[1];
...
measure q[1] -> c[0];
measure q[2] -> c[1];

written to ry_rus_round.qasm (187 lines)


## What each measurement outcome does

Outcome `00` is the target. The others apply a different but exactly known rotation,
which is why nothing is wasted: subtract it and go again.

In [21]:
print(' c[1]c[0]   probability     exact rotation?   angle applied')
for m, p, isrot, ang in U.branches(res['gates']):
    tag = '  <- success' if m == 0 else ''
    print('    %s      %-14s  %-5s          %+.9f%s'
          % (format(m, '02b'), '%.6f' % float(p), isrot, ang, tag))
print('\ntarget angle %.9f  (mod 4*pi)' % (THETA % (4*math.pi)))

 c[1]c[0]   probability     exact rotation?   angle applied
    00      0.625000        True           +3.785093762  <- success
    01      0.312500        True           +0.927295218
    10      0.062500        True           +0.000000000

target angle 3.786589509  (mod 4*pi)


## Verification

Exact integer checks: the gate set, $MM^{T}=2^{k}I$, the columns, and that every
branch is a true rotation ($A=D$, $B=-C$ as integers, not within a tolerance).

In [22]:
for key, val in U.verify(res, THETA).items():
    print('  %-28s %s' % (key, val))

  gate_set_ok                  True
  orthogonal                   True
  columns_exact                True
  all_branches_are_rotations   True
  probabilities_sum_to_1       True
  p_success                    0.625
  success_angle                3.785093762383078
  angle_error                  0.0014957467271692337
  within_eps                   True
  success_branch_unitary       True


In [23]:
# success probability must not depend on the data qubit state
M, k = R.circuit_matrix(res['gates'])
Um = np.array([[M[i][j] / math.sqrt(2)**k for j in range(8)] for i in range(8)])
dev = 0.0
for _ in range(200):
    psi = np.array([rng.gauss(0,1), rng.gauss(0,1)]); psi /= np.linalg.norm(psi)
    inp = np.zeros(8); inp[0], inp[1] = psi
    out = Um @ inp
    dev = max(dev, abs(out[0]**2 + out[1]**2 - float(res['p_success'])))
print('max deviation of P(success) over 200 random inputs: %.2e' % dev)

# and the success branch, renormalised, is exactly R_y of the achieved angle
n = math.sqrt(a*a + b*b)
K = np.array([[M[0][0], M[0][1]], [M[1][0], M[1][1]]]) / n
ang = 2*math.atan2(b, a)
print('success branch   :', np.round(K, 12).tolist())
print('R_y(%.9f):' % ang,
      [[round(math.cos(ang/2), 12), round(-math.sin(ang/2), 12)],
       [round(math.sin(ang/2), 12), round(math.cos(ang/2), 12)]])

max deviation of P(success) over 200 random inputs: 4.44e-16
success branch   : [[-0.316227766017, -0.948683298051], [0.948683298051, -0.316227766017]]
R_y(3.785093762): [[-0.316227766017, -0.948683298051], [0.948683298051, -0.316227766017]]


## The adaptive loop

Each round synthesises the *remaining* angle, samples an outcome from the exact branch
distribution, and on a non-success outcome subtracts the rotation that was applied.

In [26]:
run = U.run_adaptive(THETA, EPS, random.Random(1), pmin=PMIN)
print('rounds used   :', run['rounds'])
print('gates used    :', run['gates'], ' (CCX %d)' % run['ccx'])
print('final error   : %.3e' % run['angle_error'])
print('\nper round:')
for i, r in enumerate(run['rounds_detail']):
    print('  round %d: theta=%+.9f  k=%2d  gates=%4d  p=%.4f'
          % (i, r['theta'], r['k'], r['counts']['total'], float(r['p_success'])))

rounds used   : 1
gates used    : 50  (CCX 24)
final error   : 1.496e-03

per round:
  round 0: theta=+3.786589509  k= 4  gates=  50  p=0.6250


In [27]:
import statistics as st
cache, rng2 = {}, random.Random(11)
runs = [U.run_adaptive(THETA, EPS, rng2, pmin=PMIN, cache=cache) for _ in range(300)]
ec = U.expected_cost(THETA, EPS, pmin=PMIN)
print('300 simulated runs')
print('  mean rounds  : %.3f   (predicted 1/p = %.3f)' % (
      st.mean(r['rounds'] for r in runs), ec['expected_rounds']))
print('  mean gates   : %.1f   (predicted %.1f)' % (
      st.mean(r['gates'] for r in runs), ec['expected_gates']))
print('  worst error  : %.2e  (budget %.1e)' % (
      max(r['angle_error'] for r in runs), EPS))
print('  rounds histogram:', {n: sum(1 for r in runs if r['rounds'] == n)
                              for n in sorted({r['rounds'] for r in runs})})

300 simulated runs
  mean rounds  : 1.553   (predicted 1/p = 1.600)
  mean gates   : 83.1   (predicted 80.0)
  worst error  : 7.33e-03  (budget 1.0e-02)
  rounds histogram: {1: 183, 2: 82, 3: 23, 4: 10, 5: 2}


## Deterministic vs repeat-until-success

In [30]:
det = R.synthesize_ry(THETA, EPS)
print('                        k    gates     CCX')
print('deterministic      %6d   %6d  %6d' % (det['k'], det['counts']['total'],
                                             det['counts'].get('CCX', 0)))
print('RUS, one round     %6d   %6d  %6d' % (res['k'], res['counts']['total'],
                                             res['counts'].get('CCX', 0)))
print('RUS, expected      %6s   %6.0f  %6.0f' % ('-', ec['expected_gates'],
                                                 ec['expected_ccx']))
print('\nspeedup: %.2fx gates, %.2fx CCX'
      % (det['counts']['total']/ec['expected_gates'],
         det['counts'].get('CCX',0)/ec['expected_ccx']))

                        k    gates     CCX
deterministic          20      171      87
RUS, one round          4       50      24
RUS, expected           -       80      38

speedup: 2.14x gates, 2.27x CCX


In [31]:
# scan over precisions
print('  eps     det k  det gates | rus k  per round  p_succ  expected  speedup')
for e in range(2, 11):
    ep = 10.0**-e
    d  = R.synthesize_ry(THETA, ep)
    c  = U.expected_cost(THETA, ep, pmin=PMIN)
    print('  1e-%-2d   %5d  %8d  | %5d  %8d   %.3f  %8.0f    %.2fx'
          % (e, d['k'], d['counts']['total'], c['k'], c['gates_per_round'],
             c['p_success'], c['expected_gates'],
             d['counts']['total']/c['expected_gates']))

  eps     det k  det gates | rus k  per round  p_succ  expected  speedup
  1e-2       20       171  |     4        50   0.625        80    2.14x
  1e-3       32       279  |    16       151   0.994       152    1.84x
  1e-4       41       357  |    18       160   0.777       206    1.73x
  1e-5       49       413  |    18       164   0.682       241    1.72x
  1e-6       64       556  |    18       164   0.682       241    2.31x
  1e-7       72       605  |    29       249   0.971       256    2.36x
  1e-8       80       632  |    30       227   0.987       230    2.75x
  1e-9       87       677  |    31       284   0.533       533    1.27x
  1e-10     101       843  |    34       286   0.604       473    1.78x


## Notes

* Ancillas must be **reset** to $|00\rangle$ between rounds; conditioned on the outcome
  that is one or two $X$ gates, still inside the gate set. `run_adaptive` charges 2 gates
  per retry for this.
* $P(\text{success})$ is independent of the data qubit state, so a non-success outcome
  reveals nothing about the data and does not collapse it — it only rotates it.
* The successful branch is *exactly* unitary. Its error is a pure angle error, unlike the
  deterministic circuit which leaks $O(\varepsilon)$ amplitude into the ancillas.
* Compare modulo $4\pi$, not $2\pi$: $R_y(\theta+2\pi)=-R_y(\theta)$ and that real sign is
  tracked exactly here.
* $R_x$ is still unreachable. Real gates give real Kraus operators on every branch, so
  measurement does not help; that needs a permanent extra qubit encoding $i$.